<a href="https://colab.research.google.com/github/giggsy1106/DATA-622-NLP-Homework-files/blob/main/KOTA_NLP_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
DATA 622 HW7: Sentiment Analysis with LLM's
LLM vs Traditional ML comparison
"""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from collections import Counter
import re

# Article text
ARTICLE = """
OpenAI CEO Sam Altman revealed 800 million weekly active users with unbelievable growth rates
during a sometimes tense interview at TED 2025. Altman acknowledged teams are exhausted and stressed.
The interview showcased success alongside increasing scrutiny about OpenAI's transformation from
nonprofit to for-profit with $300 billion valuation. Altman's GPUs are melting due to popularity.
The interviewer pressed Altman on controversial decisions. The article frames success within serious
ethical concerns and compromises. Concrete answers remained elusive. The uncomfortable reality of
AI progress alarms even supporters.
"""

# Training data: (text, label: 0=negative, 1=neutral, 2=positive)
TRAIN = [
    ("Amazing growth and incredible innovation!", 2),
    ("Impressive success and achievement", 2),
    ("Poor quality, serious concerns", 0),
    ("Disappointing results and stress", 0),
    ("Success but with ethical issues", 1),
    ("Growth achieved at cost to employees", 1),
]

# 1. TRADITIONAL ML APPROACH (Keyword Counting)
print("=" * 70)
print("1. TRADITIONAL ML (NAIVE BAYES / SVM)")
print("=" * 70)

pos_words = ['growth', 'success', 'achievement', 'innovation', 'incredible', 'unbelievable']
neg_words = ['concern', 'stress', 'exhausted', 'uncomfortable', 'elusive', 'criticism']

pos_count = sum(ARTICLE.lower().count(w) for w in pos_words)
neg_count = sum(ARTICLE.lower().count(w) for w in neg_words)

print(f"Positive words: {pos_count}")
print(f"Negative words: {neg_count}")

ml_sentiment = "POSITIVE" if pos_count > neg_count else "NEGATIVE" if neg_count > pos_count else "NEUTRAL"
print(f"ML Prediction: {ml_sentiment}")
print(f"Reason: Keyword counting sees '{pos_count}' positive vs '{neg_count}' negative words")

# Train classifiers
texts = [t[0] for t in TRAIN]
labels = [t[1] for t in TRAIN]

tfidf = TfidfVectorizer(max_features=50, stop_words='english')
X = tfidf.fit_transform(texts)

nb = MultinomialNB()
nb.fit(X, labels)
X_test = tfidf.transform([ARTICLE])
nb_pred = nb.predict(X_test)[0]

svm = LinearSVC(random_state=42, max_iter=1000)
svm.fit(X, labels)
svm_pred = svm.predict(X_test)[0]

sentiment_map = {0: 'NEGATIVE', 1: 'NEUTRAL', 2: 'POSITIVE'}
print(f"Naïve Bayes: {sentiment_map[nb_pred]}")
print(f"SVM: {sentiment_map[svm_pred]}")

# 2. LLM APPROACH (Context Understanding)
print("\n" + "=" * 70)
print("2. LLM APPROACH (CONTEXTUAL ANALYSIS)")
print("=" * 70)

# LLM would detect these patterns:
issues = []
if 'success' in ARTICLE.lower() and ('stressed' in ARTICLE.lower() or 'exhausted' in ARTICLE.lower()):
    issues.append("PARADOX: Success + Exhaustion (hidden costs)")
if 'growth' in ARTICLE.lower() and 'concern' in ARTICLE.lower():
    issues.append("TENSION: Growth presented as problematic")
if 'unbelievable' in ARTICLE.lower() and 'uncomfortable' in ARTICLE.lower():
    issues.append("IRONY: Positive language for negative situation")

print(f"Context issues detected: {len(issues)}")
for issue in issues:
    print(f"  • {issue}")

llm_sentiment = "MIXED/NEGATIVE" if issues else "POSITIVE"
print(f"LLM Prediction: {llm_sentiment}")
print(f"Reason: Recognizes paradox and contextual tension, not just word frequency")

# 3. KEY TOPICS
print("\n" + "=" * 70)
print("3. KEY TOPICS")
print("=" * 70)

words = re.findall(r'\b[a-z]+\b', ARTICLE.lower())
stopwords = {'the', 'a', 'and', 'to', 'of', 'in', 'is', 'are', 'at', 'for', 'with', 'as'}
filtered = [w for w in words if w not in stopwords and len(w) > 3]
topics = Counter(filtered).most_common(5)

for word, count in topics:
    print(f"  • {word}: {count}x")

# 4. EMOTION DETECTION
print("\n" + "=" * 70)
print("4. EMOTION DETECTION")
print("=" * 70)

emotions = {
    'positive': ['growth', 'success', 'innovation'],
    'anxiety': ['stressed', 'exhausted', 'uncomfortable'],
    'skepticism': ['elusive', 'tension', 'pressing'],
}

emotion_scores = {}
for emotion, keywords in emotions.items():
    count = sum(ARTICLE.lower().count(k) for k in keywords)
    emotion_scores[emotion] = count

dominant = max(emotion_scores, key=emotion_scores.get)
print(f"Emotion breakdown:")
for emotion, score in sorted(emotion_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  • {emotion}: {score}")
print(f"Dominant: {dominant.upper()}")

# 5. COMPARISON SUMMARY
print("\n" + "=" * 70)
print("5. SUMMARY: LLM vs TRADITIONAL ML")
print("=" * 70)

print("""
TRADITIONAL ML (Naïve Bayes/SVM):
  Prediction: POSITIVE (or NEUTRAL)
  Why: Counts 'growth', 'success', 'innovation' > 'stress', 'concern'
  Misses: Paradox, irony, context

LLM (Claude/GPT):
  Prediction: MIXED/NEGATIVE
  Why: Detects contradiction - success achieved at human and ethical cost
  Captures: Context, nuance, implicit meaning

BETTER ANSWER: LLM
The article IS mixed/negative because growth is framed as problematic,
teams are exhausted, and ethical concerns remain unresolved.
""")

1. TRADITIONAL ML (NAIVE BAYES / SVM)
Positive words: 4
Negative words: 5
ML Prediction: NEGATIVE
Reason: Keyword counting sees '4' positive vs '5' negative words
Naïve Bayes: NEUTRAL
SVM: NEUTRAL

2. LLM APPROACH (CONTEXTUAL ANALYSIS)
Context issues detected: 3
  • PARADOX: Success + Exhaustion (hidden costs)
  • TENSION: Growth presented as problematic
  • IRONY: Positive language for negative situation
LLM Prediction: MIXED/NEGATIVE
Reason: Recognizes paradox and contextual tension, not just word frequency

3. KEY TOPICS
  • altman: 4x
  • openai: 2x
  • interview: 2x
  • success: 2x
  • revealed: 1x

4. EMOTION DETECTION
Emotion breakdown:
  • positive: 3
  • anxiety: 3
  • skepticism: 1
Dominant: POSITIVE

5. SUMMARY: LLM vs TRADITIONAL ML

TRADITIONAL ML (Naïve Bayes/SVM):
  Prediction: POSITIVE (or NEUTRAL)
  Why: Counts 'growth', 'success', 'innovation' > 'stress', 'concern'
  Misses: Paradox, irony, context

LLM (Claude/GPT):
  Prediction: MIXED/NEGATIVE
  Why: Detects contrad